In [1]:
pip install torch>=1.12.0 transformers>=4.20.0 gradio>=3.0 matplotlib>=3.3.0


Note: you may need to restart the kernel to use updated packages.


In [1]:
# Imports
import time
import torch
import matplotlib.pyplot as plt
from collections import Counter

from transformers import T5Tokenizer, T5ForConditionalGeneration
import gradio as gr


In [3]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
model.eval()


Using device: cpu


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [5]:
def summarize_and_visualize(text, max_length, num_beams):
    if not text.strip():
        return "⚠️ Please enter some text to summarize.", None, None

    # Prepare input
    input_str = "summarize: " + text.strip()
    inputs = tokenizer(
        input_str,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)

    # Generate summary
    start = time.time()
    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=int(max_length),
            num_beams=int(num_beams),
            early_stopping=True
        )
    end = time.time()

    # Decode and compute metrics
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    runtime = end - start
    orig_tokens = input_ids.numel()
    summ_tokens = summary_ids.numel()

    # Word frequency analysis
    words = summary.lower().split()
    common = Counter(words).most_common(10)
    labels, freqs = zip(*common) if common else ([], [])

    # Plot 1: Token count
    plt.figure(figsize=(5, 4))
    plt.bar(["Original", "Summary"], [orig_tokens, summ_tokens])
    plt.ylabel("Tokens")
    plt.title("Token Count: Original vs. Summary")
    plt.tight_layout()
    bar_chart = plt.gcf()

    # Plot 2: Word frequency
    plt.figure(figsize=(6, 4))
    plt.barh(labels, freqs)
    plt.xlabel("Frequency")
    plt.title("Top 10 Words in Summary")
    plt.tight_layout()
    freq_chart = plt.gcf()

    # Summary markdown
    summary_md = (
        f"**Summary:**\n\n{summary}\n\n"
        f"⏱ Time: {runtime:.2f}s   |   "
        f"Tokens in: {orig_tokens} → out: {summ_tokens}"
    )

    return summary_md, bar_chart, freq_chart


In [7]:
iface = gr.Interface(
    fn=summarize_and_visualize,
    inputs=[
        gr.Textbox(lines=8, label="Input Text", placeholder="Paste text here…"),
        gr.Slider(10, 200, step=10, value=60, label="Max Summary Length"),
        gr.Slider(1, 10, step=1, value=4, label="Beam Width")
    ],
    outputs=[
        gr.Markdown(label="Generated Summary"),
        gr.Plot(label="Token Count Chart"),
        gr.Plot(label="Word Frequency Chart")
    ],
    title="📝 T5 Summarizer + Visualizations",
    description="Enter text, adjust summary length & beam width, then view the summary along with token count and word-frequency charts.",
    allow_flagging="never"
)


C:\Users\koust\anaconda3\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated.Use `flagging_mode` instead.
  warnings.warn(


In [9]:
if __name__ == "__main__":
    iface.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
